# dicom_to_parquet Demo.ver: 3가지 출력 방법

`dicom_to_parquet.py`를 사용하여 DICOM 헤더 메타데이터를 추출하는 **3가지 방법**을 보여줍니다.  
원하는 방법 하나만 골라 독립적으로 실행할 수 있습니다.

| 방법 | 설명 | 출력 형태 |
|------|------|-----------|
| **방법 1** | Parquet only | `modality/tag` Hive 파티션 구조 |
| **방법 2** | Parquet → CSV | Parquet 저장 후 단일 CSV로 병합 |
| **방법 3** | CSV only | Parquet 없이 단일 CSV 직접 저장 |

**실행 순서**
1. 아래 **[Common] Import** 셀을 먼저 실행합니다.
2. 원하는 방법의 **경로 설정** 셀부터 아래로 순서대로 실행합니다.

In [7]:
# ════════════════════════════════════════════════
# [Common] Import — 세 방법 모두 이 셀을 먼저 실행하세요
# ════════════════════════════════════════════════

import os, sys

# dicom_to_parquet.py 가 있는 폴더로 작업 디렉토리 변경 (이 노트북과 같은 위치)
SCRIPT_DIR = r"C:\Projects\dicom_heterogeneity\dicom_to_parquet"
%cd $SCRIPT_DIR
sys.path.insert(0, SCRIPT_DIR)

from dicom_to_parquet import (
    build_parquet,
    build_csv_direct,
    merge_to_csv,
    iter_dicom_files,
    ARROW_SCHEMA,
)

import pyarrow.dataset as pad
import pandas as pd
from IPython.display import display

print("Import completed")

C:\Projects\dicom_heterogeneity\dicom_to_parquet
Import completed


c:\Projects\dicom_heterogeneity\dcm_venv\Lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


---
## 방법 1: Parquet only

DICOM 헤더를 `modality=.../tag=.../part-b0-0.parquet` 구조의 **Hive 파티션 Parquet**으로 저장합니다.

- 대용량 데이터셋에 가장 적합 — 쿼리 시 필요한 파티션만 읽음
- pyarrow, DuckDB, Spark 등과 바로 호환
- 필요 시 `merge_to_csv()`로 나중에 CSV 변환 가능

In [8]:
# ════════════════════════════════════════════════
# [방법 1] 경로 및 옵션 설정 — 이 셀만 수정하세요
# ════════════════════════════════════════════════

# 입력: .dcm 파일이 있는 최상위 폴더 (하위 폴더 재귀 검색)
M1_DICOM_ROOT = r"C:\Projects\dicom_data\data\mimic-iv-echo"

# 출력: Parquet 파티션을 저장할 폴더
M1_OUT_DIR    = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\01_parquet_only"

# 옵션
M1_SHARD_ROWS      = 2_000_000  # 파티션당 최대 행 수
M1_SKIP_PIXEL_DATA = True       # Pixel Data (7FE0,0010) 제외 권장
M1_EXTRA_SKIP_TAGS = []         # 추가 제외 태그 (예: ['00209999'])
M1_VERBOSE_EVERY   = 1          # N 파일마다 진행 출력 (대규모 데이터셋은 1000+ 권장)

# 입력 파일 수 미리 확인
dcm_files = list(iter_dicom_files(M1_DICOM_ROOT))
print(f"DICOM 파일 수: {len(dcm_files)}")
print(f"출력 경로    : {M1_OUT_DIR}")

DICOM 파일 수: 5
출력 경로    : C:\Projects\dicom_data\data\mimic-iv-echo_headers\01_parquet_only


In [9]:
# [방법 1] 실행
build_parquet(
    dicom_root      = M1_DICOM_ROOT,
    out_dir         = M1_OUT_DIR,
    max_files       = None,
    shard_rows      = M1_SHARD_ROWS,
    skip_pixel_data = M1_SKIP_PIXEL_DATA,
    extra_skip_tags = M1_EXTRA_SKIP_TAGS,
    verbose_every   = M1_VERBOSE_EVERY,
    export_csv      = None,
)

[INFO] files=1 bad=0 rows_buffer=62 rows_total=62
[INFO] files=2 bad=0 rows_buffer=136 rows_total=136
[INFO] files=3 bad=0 rows_buffer=215 rows_total=215
[INFO] files=4 bad=0 rows_buffer=289 rows_total=289
[INFO] files=5 bad=0 rows_buffer=351 rows_total=351
[DONE]
{
  "dicom_root": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo",
  "out_dir": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo_headers\\01_parquet_only",
  "files_processed": 5,
  "files_failed": 0,
  "rows_total": 351,
  "batches_written": 1,
  "skip_pixel_data": true,
  "skip_tags": [
    "54001010",
    "7FE00010"
  ],
  "compression": "zstd",
  "schema": [
    "file_path:string",
    "study_uid:string",
    "series_uid:string",
    "instance_uid:string",
    "sop_class_uid:string",
    "modality:string",
    "tag:string",
    "vr:string",
    "vm:string",
    "path:string",
    "value:string"
  ]
}


In [10]:
# [방법 1] 결과 확인

dataset1 = pad.dataset(M1_OUT_DIR, format="parquet", partitioning="hive")
df1 = dataset1.to_table().to_pandas()

print(f"총 행 수      : {len(df1):,}")
print(f"고유 태그 수  : {df1['tag'].nunique():,}")
print(f"Modality 종류 : {sorted(df1['modality'].dropna().unique())}")

# Private tag 포함 여부 확인 (group 번호가 홀수인 태그)
priv1 = df1[df1['tag'].apply(lambda t: int(t[:4], 16) % 2 == 1)]
print(f"Private tag 행 수: {len(priv1):,}")
if not priv1.empty:
    display(priv1[['tag','vr','path','value']].drop_duplicates('tag').head(5))

print("\n── 전체 데이터 미리보기 (10행) ──")
display(df1.head(10))

print("\n── 생성된 파티션 파일 ──")
for root, _, files in os.walk(M1_OUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        rel = os.path.relpath(full, M1_OUT_DIR)
        print(f"  {rel:55s}  {size_kb:6.1f} KB")

총 행 수      : 351
고유 태그 수  : 79
Modality 종류 : ['US']
Private tag 행 수: 0

── 전체 데이터 미리보기 (10행) ──


,file_path,study_uid,series_uid,instance_uid,sop_class_uid,vr,vm,path,value,modality,tag
0,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
1,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
2,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
3,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
4,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
5,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...,US,00080018
6,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.69855423628295450049911...,US,00080018
7,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.59351117373448806027346...,US,00080018
8,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.27493956172437740663967...,US,00080018
9,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.12795021409745600246926...,US,00080018



── 생성된 파티션 파일 ──
  _build_summary.json                                         0.6 KB
  modality=US\tag=00080016\part-b0-0.parquet                  5.1 KB
  modality=US\tag=00080018\part-b0-0.parquet                  5.4 KB
  modality=US\tag=00080020\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080021\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080023\part-b0-0.parquet                  5.0 KB
  modality=US\tag=0008002A\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080030\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080031\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080033\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080050\part-b0-0.parquet                  4.9 KB
  modality=US\tag=00080060\part-b0-0.parquet                  4.9 KB
  modality=US\tag=00080070\part-b0-0.parquet                  5.1 KB
  modality=US\tag=00080080\part-b0-0.parquet                  5.1 KB
  modality=US\ta

---
## 방법 2: Parquet 저장 후 CSV로 병합

방법 1과 동일하게 Parquet 파티션을 먼저 저장한 뒤, **모든 파티션을 단일 flat CSV로 병합**합니다.

- 결과물 2개: Parquet 파티션 + `all_tags.csv`
- 이미 Parquet이 있다면 `merge_to_csv()`만 단독 호출 가능
- (주의) 대용량 데이터셋에서는 CSV 병합 단계에 시간·용량이 많이 소요될 수 있음

In [11]:
# ════════════════════════════════════════════════
# [방법 2] 경로 및 옵션 설정 — 이 셀만 수정하세요
# ════════════════════════════════════════════════

# 입력: .dcm 파일이 있는 최상위 폴더
M2_DICOM_ROOT = r"C:\Projects\dicom_data\data\mimic-iv-echo"

# 출력 ①: Parquet 파티션 저장 폴더
M2_OUT_DIR    = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv"

# 출력 ②: 병합 CSV 경로 (Parquet 완성 후 자동 생성)
M2_EXPORT_CSV = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv\all_tags.csv"

# 옵션
M2_SHARD_ROWS      = 2_000_000
M2_SKIP_PIXEL_DATA = True
M2_EXTRA_SKIP_TAGS = []
M2_VERBOSE_EVERY   = 1

dcm_files = list(iter_dicom_files(M2_DICOM_ROOT))
print(f"DICOM 파일 수 : {len(dcm_files)}")
print(f"Parquet 출력  : {M2_OUT_DIR}")
print(f"CSV 출력      : {M2_EXPORT_CSV}")

DICOM 파일 수 : 5
Parquet 출력  : C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv
CSV 출력      : C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv\all_tags.csv


In [12]:
# [방법 2] 실행: Parquet 저장 → CSV 자동 병합
build_parquet(
    dicom_root      = M2_DICOM_ROOT,
    out_dir         = M2_OUT_DIR,
    max_files       = None,
    shard_rows      = M2_SHARD_ROWS,
    skip_pixel_data = M2_SKIP_PIXEL_DATA,
    extra_skip_tags = M2_EXTRA_SKIP_TAGS,
    verbose_every   = M2_VERBOSE_EVERY,
    export_csv      = M2_EXPORT_CSV,   # 이 경로를 지정하면 Parquet 완성 후 자동으로 CSV 병합
)

[INFO] files=1 bad=0 rows_buffer=62 rows_total=62
[INFO] files=2 bad=0 rows_buffer=136 rows_total=136
[INFO] files=3 bad=0 rows_buffer=215 rows_total=215
[INFO] files=4 bad=0 rows_buffer=289 rows_total=289
[INFO] files=5 bad=0 rows_buffer=351 rows_total=351
[DONE]
{
  "dicom_root": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo",
  "out_dir": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo_headers\\02_parquet_and_csv",
  "files_processed": 5,
  "files_failed": 0,
  "rows_total": 351,
  "batches_written": 1,
  "skip_pixel_data": true,
  "skip_tags": [
    "54001010",
    "7FE00010"
  ],
  "compression": "zstd",
  "schema": [
    "file_path:string",
    "study_uid:string",
    "series_uid:string",
    "instance_uid:string",
    "sop_class_uid:string",
    "modality:string",
    "tag:string",
    "vr:string",
    "vm:string",
    "path:string",
    "value:string"
  ]
}
[CSV] Scanning partitions in C:\Projects\dicom_data\data\mimic-iv-echo_headers\02_parquet_and_csv ...
[CSV] Done: 351 rows

[CSV] 5 rows written ...
[CSV] 10 rows written ...
[CSV] 15 rows written ...
[CSV] 20 rows written ...
[CSV] 25 rows written ...
[CSV] 30 rows written ...
[CSV] 35 rows written ...
[CSV] 40 rows written ...
[CSV] 45 rows written ...
[CSV] 50 rows written ...
[CSV] 55 rows written ...
[CSV] 60 rows written ...
[CSV] 65 rows written ...
[CSV] 70 rows written ...
[CSV] 75 rows written ...
[CSV] 80 rows written ...
[CSV] 81 rows written ...
[CSV] 86 rows written ...
[CSV] 91 rows written ...
[CSV] 93 rows written ...
[CSV] 95 rows written ...
[CSV] 97 rows written ...
[CSV] 102 rows written ...
[CSV] 107 rows written ...
[CSV] 112 rows written ...
[CSV] 117 rows written ...
[CSV] 122 rows written ...
[CSV] 127 rows written ...
[CSV] 132 rows written ...
[CSV] 137 rows written ...
[CSV] 139 rows written ...
[CSV] 141 rows written ...
[CSV] 146 rows written ...
[CSV] 148 rows written ...
[CSV] 150 rows written ...
[CSV] 155 rows written ...
[CSV] 157 rows written ...
[CSV] 159 rows written .

In [13]:
# [방법 2] 결과 확인

# Parquet 결과 (exclude_invalid_files=True: 동일 폴더의 CSV 등 non-parquet 파일 무시)
dataset2 = pad.dataset(M2_OUT_DIR, format="parquet", partitioning="hive", exclude_invalid_files=True)
df2_parquet = dataset2.to_table().to_pandas()
print(f"[Parquet] 총 행 수: {len(df2_parquet):,}")

# CSV 결과
df2_csv = pd.read_csv(M2_EXPORT_CSV)
csv2_size_mb = os.path.getsize(M2_EXPORT_CSV) / 1024 / 1024
print(f"[CSV]     총 행 수: {len(df2_csv):,}")
print(f"[CSV]     파일 크기: {csv2_size_mb:.3f} MB")

# Parquet 행 수와 CSV 행 수 일치 검증
if len(df2_parquet) == len(df2_csv):
    print("OK: Parquet 행 수 == CSV 행 수 — 병합 정상")
else:
    print(f"WARNING: 행 수 불일치 — Parquet {len(df2_parquet):,} / CSV {len(df2_csv):,}")

print("\n── CSV 미리보기 (10행) ──")
display(df2_csv.head(10))

print("\n── 생성된 파일 목록 ──")
for root, _, files in os.walk(M2_OUT_DIR):
    for f in sorted(files):
        full = os.path.join(root, f)
        size_kb = os.path.getsize(full) / 1024
        rel = os.path.relpath(full, M2_OUT_DIR)
        print(f"  {rel:55s}  {size_kb:6.1f} KB")

[Parquet] 총 행 수: 351
[CSV]     총 행 수: 351
[CSV]     파일 크기: 0.114 MB
OK: Parquet 행 수 == CSV 행 수 — 병합 정상

── CSV 미리보기 (10행) ──


,file_path,study_uid,series_uid,instance_uid,sop_class_uid,vr,vm,path,value,modality,tag
0,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
1,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
2,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
3,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.3.1,US,00080016
4,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1,US,00080016
5,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...,US,00080018
6,C:\Projects\dicom_data\data\mimic-iv-echo\9410...,1.2.840.113554.6.1.104.68301667710199290520777...,1.2.840.113554.6.1.105.52875607447023178069717...,1.2.840.113554.6.1.101.69855423628295450049911...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.69855423628295450049911...,US,00080018
7,C:\Projects\dicom_data\data\mimic-iv-echo\9802...,1.2.840.113554.6.1.104.12917773695659117017448...,1.2.840.113554.6.1.105.75593455491512733255625...,1.2.840.113554.6.1.101.59351117373448806027346...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.59351117373448806027346...,US,00080018
8,C:\Projects\dicom_data\data\mimic-iv-echo\9832...,1.2.840.113554.6.1.104.63742685390757424135156...,1.2.840.113554.6.1.105.97320391386664344724944...,1.2.840.113554.6.1.101.27493956172437740663967...,1.2.840.10008.5.1.4.1.1.3.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.27493956172437740663967...,US,00080018
9,C:\Projects\dicom_data\data\mimic-iv-echo\9989...,1.2.840.113554.6.1.104.86871504177171794777507...,1.2.840.113554.6.1.105.64352159466407455443550...,1.2.840.113554.6.1.101.12795021409745600246926...,1.2.840.10008.5.1.4.1.1.6.1,UI,1,00080018[0]/,1.2.840.113554.6.1.101.12795021409745600246926...,US,00080018



── 생성된 파일 목록 ──
  _build_summary.json                                         0.6 KB
  all_tags.csv                                              116.9 KB
  modality=US\tag=00080016\part-b0-0.parquet                  5.1 KB
  modality=US\tag=00080018\part-b0-0.parquet                  5.4 KB
  modality=US\tag=00080020\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080021\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080023\part-b0-0.parquet                  5.0 KB
  modality=US\tag=0008002A\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080030\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080031\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080033\part-b0-0.parquet                  5.0 KB
  modality=US\tag=00080050\part-b0-0.parquet                  4.9 KB
  modality=US\tag=00080060\part-b0-0.parquet                  4.9 KB
  modality=US\tag=00080070\part-b0-0.parquet                  5.1 KB
  modality=US\tag

---
## 방법 3: CSV only (Parquet 없이 직접 저장)

Parquet 중간 파일 없이 DICOM 헤더를 **단일 flat CSV로 바로 저장**합니다.

- 방법 1·2와 동일한 tag-explosion 로직: private tag, nested SQ, multi-value 모두 포함
- Parquet 도구 사용이 어렵거나 데이터셋이 작을 때 적합
- (주의) 대용량 데이터셋에서는 파일이 매우 커질 수 있음 (방법 1 권장)

In [14]:
# ════════════════════════════════════════════════
# [방법 3] 경로 및 옵션 설정 — 이 셀만 수정하세요
# ════════════════════════════════════════════════

# 입력: .dcm 파일이 있는 최상위 폴더
M3_DICOM_ROOT = r"C:\Projects\dicom_data\data\mimic-iv-echo"

# 출력: CSV 저장 경로 (파일명 포함)
M3_CSV_PATH   = r"C:\Projects\dicom_data\data\mimic-iv-echo_headers\03_csv_only\all_tags.csv"

# 옵션
M3_SKIP_PIXEL_DATA = True
M3_EXTRA_SKIP_TAGS = []
M3_VERBOSE_EVERY   = 1
M3_BUFFER_ROWS     = 100_000  # 이 행 수마다 디스크에 flush (메모리 사용량 조절)

os.makedirs(os.path.dirname(M3_CSV_PATH), exist_ok=True)

dcm_files = list(iter_dicom_files(M3_DICOM_ROOT))
print(f"DICOM 파일 수: {len(dcm_files)}")
print(f"CSV 출력 경로: {M3_CSV_PATH}")

DICOM 파일 수: 5
CSV 출력 경로: C:\Projects\dicom_data\data\mimic-iv-echo_headers\03_csv_only\all_tags.csv


In [15]:
# [방법 3] 실행: CSV 직접 저장 (Parquet 없음)
build_csv_direct(
    dicom_root      = M3_DICOM_ROOT,
    csv_path        = M3_CSV_PATH,
    max_files       = None,
    skip_pixel_data = M3_SKIP_PIXEL_DATA,
    extra_skip_tags = M3_EXTRA_SKIP_TAGS,
    verbose_every   = M3_VERBOSE_EVERY,
    buffer_rows     = M3_BUFFER_ROWS,
)

[INFO] files=1 bad=0 rows=62
[INFO] files=2 bad=0 rows=136
[INFO] files=3 bad=0 rows=215
[INFO] files=4 bad=0 rows=289
[INFO] files=5 bad=0 rows=351
[DONE]
{
  "dicom_root": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo",
  "csv_path": "C:\\Projects\\dicom_data\\data\\mimic-iv-echo_headers\\03_csv_only\\all_tags.csv",
  "files_processed": 5,
  "files_failed": 0,
  "rows_total": 351
}


In [16]:
# [방법 3] 결과 확인

df3 = pd.read_csv(M3_CSV_PATH)
csv3_size_mb = os.path.getsize(M3_CSV_PATH) / 1024 / 1024

print(f"총 행 수      : {len(df3):,}")
print(f"파일 크기     : {csv3_size_mb:.3f} MB")
print(f"고유 태그 수  : {df3['tag'].nunique():,}")

# Private tag 포함 여부 확인
priv3 = df3[df3['tag'].apply(lambda t: int(t[:4], 16) % 2 == 1)]
print(f"Private tag 행 수: {len(priv3):,}")
if not priv3.empty:
    display(priv3[['tag','vr','path','value']].drop_duplicates('tag').head(5))

# Nested SQ 경로 확인 (path에 '[숫자]' 포함 = sequence 내부 element)
sq3 = df3[df3['path'].str.contains(r'\[\d+\]', regex=True)]
print(f"Nested SQ 행 수 : {len(sq3):,}")
if not sq3.empty:
    display(sq3[['tag','vr','path','value']].head(5))

print("\n── CSV 미리보기 (10행) ──")
display(df3.head(10))

총 행 수      : 351
파일 크기     : 0.114 MB
고유 태그 수  : 79
Private tag 행 수: 0
Nested SQ 행 수 : 346


,tag,vr,path,value
0,00080016,UI,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1
1,00080018,UI,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...
2,00080020,DA,00080020[0]/,21510406
3,00080021,DA,00080021[0]/,21510406
4,00080023,DA,00080023[0]/,21510406



── CSV 미리보기 (10행) ──


,file_path,study_uid,series_uid,instance_uid,sop_class_uid,modality,tag,vr,vm,path,value
0,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080016,UI,1,00080016[0]/,1.2.840.10008.5.1.4.1.1.6.1
1,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080018,UI,1,00080018[0]/,1.2.840.113554.6.1.101.50408064018308145405200...
2,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080020,DA,1,00080020[0]/,21510406
3,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080021,DA,1,00080021[0]/,21510406
4,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080023,DA,1,00080023[0]/,21510406
5,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,0008002A,DT,1,0008002A[0]/,21510406083003
6,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080030,TM,1,00080030[0]/,081600
7,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080031,TM,1,00080031[0]/,081600
8,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080033,TM,1,00080033[0]/,083003
9,C:\Projects\dicom_data\data\mimic-iv-echo\9210...,1.2.840.113554.6.1.104.13126863208603407030059...,1.2.840.113554.6.1.105.97571479085449298347219...,1.2.840.113554.6.1.101.50408064018308145405200...,1.2.840.10008.5.1.4.1.1.6.1,US,00080050,SH,0,00080050[0]/,NaN


---
## [선택] 세 방법 결과 비교

세 방법을 모두 실행한 경우에만 이 셀들을 실행하세요.

In [17]:
# [선택] 세 방법 결과 종합 비교
# df1, df2_csv, df3 가 모두 정의된 경우에만 정상 동작

def folder_size_mb(path):
    # 폴더 전체 크기 (MB) 계산
    total = sum(
        os.path.getsize(os.path.join(r, f))
        for r, _, files in os.walk(path)
        for f in files
    )
    return total / 1024 / 1024

def count_private(df):
    # group 번호가 홀수인 태그 행 수
    return len(df[df['tag'].apply(lambda t: int(t[:4], 16) % 2 == 1)])

summary = pd.DataFrame([
    {
        "방법": "방법 1: Parquet only",
        "총 행 수": len(df1),
        "고유 태그": df1['tag'].nunique(),
        "Private tag 행": count_private(df1),
        "디스크 크기 (MB)": round(folder_size_mb(M1_OUT_DIR), 3),
        "출력 형태": "파티션 Parquet",
    },
    {
        "방법": "방법 2: Parquet + CSV",
        "총 행 수": len(df2_csv),
        "고유 태그": df2_csv['tag'].nunique(),
        "Private tag 행": count_private(df2_csv),
        "디스크 크기 (MB)": round(folder_size_mb(M2_OUT_DIR), 3),
        "출력 형태": "파티션 Parquet + CSV",
    },
    {
        "방법": "방법 3: CSV only",
        "총 행 수": len(df3),
        "고유 태그": df3['tag'].nunique(),
        "Private tag 행": count_private(df3),
        "디스크 크기 (MB)": round(folder_size_mb(os.path.dirname(M3_CSV_PATH)), 3),
        "출력 형태": "단일 CSV",
    },
])

display(summary.set_index("방법"))

counts = [len(df1), len(df2_csv), len(df3)]
if len(set(counts)) == 1:
    print(f"\nOK: 세 방법 모두 동일한 행 수 ({counts[0]:,}행)")
else:
    print(f"\nWARNING: 행 수 불일치 — {counts}")

,총 행 수,고유 태그,Private tag 행,디스크 크기 (MB),출력 형태
방법,,,,,
방법 1: Parquet only,351,79,0,0.380,파티션 Parquet
방법 2: Parquet + CSV,351,79,0,0.494,파티션 Parquet + CSV
방법 3: CSV only,351,79,0,0.114,단일 CSV



OK: 세 방법 모두 동일한 행 수 (351행)


In [18]:
# [선택] 태그 분포 분석 (방법 1 결과 기준)

print("── VR 분포 ──")
display(df1['vr'].value_counts().rename_axis('VR').reset_index(name='행 수').head(15))

print("\n── 자주 등장하는 태그 Top 20 ──")
top_tags = (
    df1.groupby('tag')
    .agg(행수=('tag','count'), VR=('vr','first'), 샘플값=('value','first'))
    .sort_values('행수', ascending=False)
    .head(20)
)
display(top_tags)

── VR 분포 ──


,VR,행 수
0,US,68
1,LO,41
2,UL,38
3,CS,33
4,IS,27
5,DS,21
6,UI,20
7,DA,20
8,TM,15
9,SH,15



── 자주 등장하는 태그 Top 20 ──


,행수,VR,샘플값
tag,,,
00186012,6,US,1
00186020,6,SL,289
0018602E,6,FD,0.0359764
0018602C,6,FD,0.0359764
00186026,6,US,3
00186024,6,US,3
00186022,6,SL,1
0018601E,6,UL,519
0018601C,6,UL,796
